In [8]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

In [9]:
train = pd.read_parquet('datasets/block1_train.parquet', engine='fastparquet')
test2 = pd.read_parquet('datasets/block2_test.parquet', engine='fastparquet')
test3 = pd.read_parquet('datasets/block3_test.parquet', engine='fastparquet')

insurer_cols = [c for c in train.columns if c.endswith('_price')]
insurers = [c.replace('_price', '') for c in insurer_cols]

print(f"Train shape: {train.shape}")
print(f"Test2 shape: {test2.shape}")
print(f"Test3 shape: {test3.shape}")
print(f"Insurers: {insurers}")

Train shape: (541292, 155)
Test2 shape: (164092, 144)
Test3 shape: (142216, 144)
Insurers: ['Insurer_A', 'Insurer_B', 'Insurer_C', 'Insurer_D', 'Insurer_E', 'Insurer_F', 'Insurer_G', 'Insurer_H', 'Insurer_I', 'Insurer_J', 'Insurer_K']


In [10]:
def engineer_features(df):
    df = df.copy()

    # Dropiraj kolone koje su beskorisne
    drop_cols = [
        'second_driver_birthdate',
        'second_driver_claim_free_years',
        'vehicle_number_plate',
        'postal_code_houses_owned_by_rental_association_ratio',
    ]
    df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)

    # Starost vozača iz datuma rođenja
    ref_date = pd.Timestamp('2024-01-01')
    if 'contractor_birthdate' in df.columns:
        df['contractor_birthdate'] = pd.to_datetime(
            df['contractor_birthdate'], errors='coerce')
        df['driver_age'] = (ref_date - df['contractor_birthdate']).dt.days / 365.25
        df.drop(columns=['contractor_birthdate'], inplace=True)

    # Starost vozila
    if 'vehicle_first_registration_date' in df.columns:
        df['vehicle_first_registration_date'] = pd.to_datetime(
            df['vehicle_first_registration_date'], errors='coerce')
        df['vehicle_age_years'] = (
            ref_date - df['vehicle_first_registration_date']).dt.days / 365.25
        df.drop(columns=['vehicle_first_registration_date'], inplace=True)

    # Risk features iz claim_free_years
    if 'claim_free_years' in df.columns:
        df['claim_free_years'] = pd.to_numeric(df['claim_free_years'], errors='coerce')
        df['is_bad_driver'] = (df['claim_free_years'] < 0).astype(int)
        df['claim_free_years_abs'] = df['claim_free_years'].abs()

    # Power-to-weight ratio vozila
    if 'vehicle_net_max_power' in df.columns and 'vehicle_weight' in df.columns:
        df['power_to_weight'] = df['vehicle_net_max_power'] / (df['vehicle_weight'] + 1)

    # Za kolone sa >80% missing — dodaj binary flag "da li podatak postoji"
    insurer_price_cols = [c for c in df.columns if c.endswith('_price')]
    high_missing = [
        c for c in df.columns
        if df[c].isnull().mean() > 0.8
        and c not in insurer_price_cols
        and 'deductible' not in c
    ]
    for col in high_missing:
        df[f'{col}_known'] = df[col].notnull().astype(int)

    # Kategoričke kolone → category dtype (LightGBM ih voli ovako)
    cat_cols = df.select_dtypes(include=['object']).columns.tolist()
    cat_cols = [c for c in cat_cols if c != 'quote_id']
    for col in cat_cols:
        df[col] = df[col].astype('category')

    return df

In [11]:
train = engineer_features(train)
test2 = engineer_features(test2)
test3 = engineer_features(test3)

print(f"Features after engineering: {train.shape[1]}")

Features after engineering: 160


In [12]:
exclude = ['quote_id'] + \
          [c for c in train.columns if c.endswith('_price')]

feature_cols = [c for c in train.columns if c not in exclude]

print(f"Number of features: {len(feature_cols)}")
print(feature_cols[:20])  # preview prvih 20

Number of features: 148
['vehicle_ownership_duration', 'coverage', 'claim_free_years', 'payment_frequency', 'is_driver_owner', 'usage', 'vehicle_maker', 'vehicle_model', 'vehicle_fuel_type', 'vehicle_engine_size', 'vehicle_power', 'vehicle_net_weight', 'vehicle_gross_weight', 'vehicle_length', 'vehicle_width', 'vehicle_height', 'vehicle_number_of_cylinders', 'vehicle_number_of_doors', 'vehicle_number_of_seats', 'vehicle_number_of_wheels']


In [13]:
lgbm_params = {
        'objective': 'regression_l1',
    'metric': 'mae',
    'learning_rate': 0.08,        # bilo 0.05 → brže konvergira
    'num_leaves': 63,             # bilo 127 → upola manje stabala
    'min_child_samples': 100,     # bilo 50 → manje splittova
    'feature_fraction': 0.7,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'n_jobs': -1,
    'verbose': -1,
    'random_state': 42,
}

In [ ]:
models = {}
oof_maes = {}
predictions_test2 = {}
predictions_test3 = {}

N_FOLDS = 3
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

for insurer in insurers:
    price_col = f'{insurer}_price'

    # Samo redovi gde insurer daje quote
    mask = train[price_col].notnull()
    X = train.loc[mask, feature_cols].copy()
    y = train.loc[mask, price_col].copy()

    # Log transform jer su cene skewed
    y_log = np.log1p(y)

    print(f"\nTraining {insurer} | rows: {mask.sum():,}")

    oof_preds = np.zeros(len(X))
    fold_maes = []
    fold_models = []

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y_log.iloc[tr_idx], y_log.iloc[val_idx]

        dtrain = lgb.Dataset(X_tr, label=y_tr)
        dval   = lgb.Dataset(X_val, label=y_val, reference=dtrain)

        model = lgb.train(
    lgbm_params,
    dtrain,
    num_boost_round=1000,          # bilo 2000
    valid_sets=[dval],
    callbacks=[
        lgb.early_stopping(stopping_rounds=30, verbose=False),  # bilo 50
        lgb.log_evaluation(period=100)
    ]
)

        val_pred = np.expm1(model.predict(X_val))
        y_val_orig = np.expm1(y_val)

        fold_mae = mean_absolute_error(y_val_orig, val_pred)
        fold_maes.append(fold_mae)
        fold_models.append(model)
        oof_preds[val_idx] = model.predict(X_val)

        print(f"  Fold {fold+1} MAE: {fold_mae:.4f}")

    oof_mae = mean_absolute_error(np.expm1(y_log), np.expm1(oof_preds))
    oof_maes[insurer] = oof_mae
    models[insurer] = fold_models
    print(f"  >>> OOF MAE: {oof_mae:.4f}")

    # Predikcija — average svih 5 foldova
    X_test2 = test2[feature_cols].copy()
    X_test3 = test3[feature_cols].copy()

    preds2 = np.mean([np.expm1(m.predict(X_test2)) for m in fold_models], axis=0)
    preds3 = np.mean([np.expm1(m.predict(X_test3)) for m in fold_models], axis=0)

    # Clip outliera
    train_min = y.min()
    train_max = y.max()
    predictions_test2[price_col] = np.clip(preds2, train_min * 0.9, train_max * 1.1)
    predictions_test3[price_col] = np.clip(preds3, train_min * 0.9, train_max * 1.1)


Training Insurer_A | rows: 527,372
[100]	valid_0's l1: 0.116208
[200]	valid_0's l1: 0.103614
[300]	valid_0's l1: 0.0998123
[400]	valid_0's l1: 0.0980991
[500]	valid_0's l1: 0.0968826
[600]	valid_0's l1: 0.096146
[700]	valid_0's l1: 0.0955209
[800]	valid_0's l1: 0.0950385
[900]	valid_0's l1: 0.094543
[1000]	valid_0's l1: 0.0941872


In [ ]:
def make_submission(test_df, preds_dict, filename):
    sub = pd.DataFrame({'quote_id': test_df['quote_id']})
    for col in insurer_cols:
        sub[col] = preds_dict[col]
    sub.to_csv(filename, sep=';', index=False, decimal='.')
    print(f"Saved: {filename} | shape: {sub.shape}")

make_submission(test2, predictions_test2, 'submission_block2.csv')
make_submission(test3, predictions_test3, 'submission_block3.csv')

In [ ]:
print("\nOOF MAE SUMMARY:")
print("-" * 30)
for ins, mae in sorted(oof_maes.items(), key=lambda x: x[1], reverse=True):
    print(f"  {ins}: {mae:.4f}")
print(f"\nOVERALL MEAN MAE: {np.mean(list(oof_maes.values())):.4f}")



OOF MAE SUMMARY:
------------------------------

OVERALL MEAN MAE: nan
